# Query Processing

In [1]:
import numpy as np
import pandas as pd

In [2]:
import nltk
from nltk.corpus import wordnet as wn
from nltk.util import ngrams
from collections import defaultdict, Counter
import re

nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/aakashshrestha/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/aakashshrestha/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/aakashshrestha/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
documents = [
    "The cat sat on the mat.",
    "Dogs are loyal pets.",
    "Cats and dogs can be friends.",
    "He owns a domestic cat."
]

query = "cat"
vocab = ["cat", "dog", "mat", "pets", "sat", "on", "loyal", "friends"]
corpus = " ".join(documents)

### Query Expansion

In [4]:
def query_expansion_wordnet(query):
    words = nltk.word_tokenize(query)
    expanded_query = set(words)
    
    for word in words:
        for syn in wn.synsets(word):
            for lemma in syn.lemmas():
                expanded_query.add(lemma.name().replace('_', ' '))
    return list(expanded_query)

print("1. Query Expansion (WordNet):")
print(query_expansion_wordnet(query))
            

1. Query Expansion (WordNet):
['computerized axial tomography', 'cat', 'African tea', 'computed axial tomography', 'retch', 'regorge', 'computed tomography', 'Caterpillar', 'quat', 'be sick', 'upchuck', 'big cat', 'guy', 'vomit', 'sick', 'chuck', 'throw up', 'disgorge', 'spue', 'puke', 'CAT', 'cast', 'CT', 'true cat', 'spew', 'honk', 'hombre', 'khat', "cat-o'-nine-tails", 'bozo', 'Arabian tea', 'regurgitate', 'vomit up', 'barf', 'qat', 'computerized tomography', 'kat', 'purge']


### Spelling Correction Techniques

In [12]:
# A. Edit Distance
def edit_distance(w1, w2):
    dp = [[0] * (len(w2)+1) for _ in range(len(w1)+1)]
    for i in range(len(w1)+1):
        for j in range(len(w2)+1):
            if i == 0:
                dp[i][j] = j
            elif j == 0:
                dp[i][j] = i
            elif w1[i-1] == w2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[-1][-1]

def correct_by_edit_distance(word, vocab):
  min_dist = float('inf')
  correction = word
  for w in vocab:
      dist = edit_distance(word, w)
      if dist < min_dist:
          min_dist = dist
          correction = w
  return correction


print("\nEdit Distance Correction of cta:")
print(correct_by_edit_distance("cta", vocab))


Edit Distance Correction of cta:
cat


In [6]:
# B. K gram
def k_grams(word, k=2):
    return {word[i:i+k] for i in range(len(word)-k+1)}

def kgram_correction(word, vocab, k=2):
    word_grams = k_grams(word, k)
    scores = []
    for v in vocab:
        v_grams = k_grams(v, k)
        overlap = len(word_grams & v_grams) / max(len(word_grams), 1)
        scores.append((v, overlap))
    return sorted(scores, key=lambda x: -x[1])[:3]

print("Spelling Correction (K-gram):", kgram_correction("ct", vocab))

Spelling Correction (K-gram): [('cat', 0.0), ('dog', 0.0), ('mat', 0.0)]


In [7]:
# C. Context Sensitive (based on co-occurrence in corpus)
def context_sensitive(word, corpus, vocab):
    tokens = corpus.split()
    context = Counter()
    for i, w in enumerate(tokens):
        if w == word and i > 0:
            context[tokens[i-1]] += 1
        if w == word and i < len(tokens)-1:
            context[tokens[i+1]] += 1
    return context.most_common(3)

print("Spelling Correction (Context-sensitive for 'cat'):", context_sensitive("cat", corpus, vocab))

Spelling Correction (Context-sensitive for 'cat'): [('The', 1), ('sat', 1)]


### Query Language Variations

In [8]:
# A. Single query
single_query = [d for d in documents if query in d.lower()]
print("\nSingle Query Results:", single_query)


Single Query Results: ['The cat sat on the mat.', 'Cats and dogs can be friends.', 'He owns a domestic cat.']


In [9]:
# B Boolean query: "cat AND dog"
bool_query = [d for d in documents if "cat" in d.lower() and "dog" in d.lower()]
print("Boolean Query (cat AND dog):", bool_query)


Boolean Query (cat AND dog): ['Cats and dogs can be friends.']


In [11]:
# C. Natural / Structured query simulation
natural_query = "Show me documents about cats or dogs being pets."
structured_query_terms = ["cat", "dog", "pets"]

structured_results = [d for d in documents if any(term in d.lower() for term in structured_query_terms)]
print("Natural Query:", natural_query)
print("Structured Query Results:")
for result in structured_results:
    print(result)

Natural Query: Show me documents about cats or dogs being pets.
Structured Query Results:
The cat sat on the mat.
Dogs are loyal pets.
Cats and dogs can be friends.
He owns a domestic cat.
